# Rheumatoid Arthritis Synovial Tissue scRNA-seq Analysis

**BioOptimize-scMultiomics Example Project**

This notebook demonstrates a realistic end-to-end analysis of publicly available single-cell RNA-seq data from rheumatoid arthritis (RA) patients.

## Dataset
- **Accession**: E-MTAB-14192 (ArrayExpress / BioStudies)
- **Source**: Single cell RNA sequencing of paired PBMC and synovial tissue cells from Rheumatoid Arthritis (RA) patients naive to treatment
- **Design**: Live CD45+ cells from peripheral blood (PB) and synovial tissue (ST) of 3 treatment-naive RA patients (10x Genomics 3' v2)
- **Link**: https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-MTAB-14192

### Data Download
Processed 10x matrices (matrix.mtx.gz, barcodes.tsv.gz, features.tsv.gz) are available for each sample (e.g., SA272_SYN, SA272_Blood, SA276_SYN, etc.).

Download from BioStudies or FTP:
```bash
# Example (replace with actual file names from the study page)
wget https://ftp.ebi.ac.uk/biostudies/fire/E-MTAB-/192/E-MTAB-14192/Files/SA272_SYN_matrix.mtx.gz
# Repeat for barcodes, features, and other samples
```

Place files in `data/raw/RA_E-MTAB-14192/` organized by sample.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white')

print('Scanpy version:', sc.__version__)

## 1. Load and Combine Samples

Load individual 10x matrices and annotate with sample and tissue metadata.

In [ ]:
# Example structure — adjust paths after downloading
data_dir = Path('data/raw/RA_E-MTAB-14192')

samples = {
    'SA272_SYN': {'tissue': 'synovium', 'patient': 'SA272'},
    'SA272_Blood': {'tissue': 'blood', 'patient': 'SA272'},
    'SA276_SYN': {'tissue': 'synovium', 'patient': 'SA276'},
    'SA276_Blood': {'tissue': 'blood', 'patient': 'SA276'},
    # Add remaining samples as needed
}

adatas = []
for sample_name, meta in samples.items():
    sample_path = data_dir / sample_name
    if sample_path.exists():
        adata = sc.read_10x_mtx(sample_path, var_names='gene_symbols', cache=True)
        adata.obs['sample'] = sample_name
        adata.obs['tissue'] = meta['tissue']
        adata.obs['patient'] = meta['patient']
        adata.obs['batch'] = sample_name  # for integration
        adatas.append(adata)
        print(f'Loaded {sample_name}: {adata.n_obs} cells')
    else:
        print(f'Skipping {sample_name} (not found)')

if adatas:
    adata = sc.concat(adatas, join='outer', label='sample', keys=list(samples.keys()))
    adata.obs_names_make_unique()
    print(f'Combined: {adata.n_obs} cells, {adata.n_vars} genes')
else:
    print('No data loaded. Download matrices first.')
    # For demo purposes, you can also start from a pre-processed h5ad if available

## 2. Quality Control

In [ ]:
# Mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# QC plots
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

# Filtering
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mt < 20, :].copy()

print(f'After QC: {adata.n_obs} cells')

## 3. Normalization, HVG, and Integration (Harmony)

Using the BioOptimize pipeline approach for multi-sample integration.

In [ ]:
# Normalization
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key='batch')
sc.pl.highly_variable_genes(adata)

adata = adata[:, adata.var.highly_variable].copy()

# Scale and PCA
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack', n_comps=30)

# Harmony integration (correct for sample/patient batch)
sc.external.pp.harmony_integrate(adata, key='batch', basis='X_pca')

# Neighbors and UMAP on integrated embedding
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca_harmony')
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.6)

## 4. Visualization and Interpretation

In [ ]:
# UMAP colored by metadata
sc.pl.umap(adata, color=['leiden', 'tissue', 'patient', 'batch'], ncols=2)

# Marker genes
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

# Known immune markers in RA context
marker_genes = ['CD3D', 'CD4', 'CD8A', 'FOXP3',  # T cells
                'CD79A', 'MS4A1',               # B cells
                'LYZ', 'CD14', 'CD68', 'C1QA',  # Myeloid / macrophages
                'NKG7', 'GNLY',                 # NK
                'COL1A1', 'PDPN']               # Fibroblasts (if present)

sc.pl.umap(adata, color=marker_genes, ncols=3)

## 5. Synovium vs Blood Comparison

Key biological question: How does the immune landscape differ between synovial tissue and peripheral blood in treatment-naive RA?

In [ ]:
# Composition by tissue
sc.pl.umap(adata, color='tissue')

# Differential abundance or simple proportions
composition = pd.crosstab(adata.obs['leiden'], adata.obs['tissue'], normalize='columns')
print(composition)

# Focus on myeloid / macrophage states (often expanded in RA synovium)
sc.pl.umap(adata, color=['CD14', 'CD68', 'C1QA', 'IL1B', 'TNF'], ncols=3)

# Save results
adata.write('results/RA_synovium_processed.h5ad')
print('Analysis complete. Results saved.')

## Summary

This example shows a complete, reproducible analysis of real RA synovial and blood scRNA-seq data:
- Multi-sample loading and annotation
- Standard QC
- Harmony integration across patients/samples
- Clustering, marker discovery, and tissue comparison

**Next steps** (for a full project):
- Annotate clusters with reference markers or automated tools
- Trajectory analysis of monocyte-to-macrophage transitions
- Ligand-receptor analysis (e.g., CellPhoneDB or LIANA)
- Integration with scATAC-seq or spatial data if available

Research use only.